# nb_generar_datos
Genera una carga masiva de facturas sintéticas con Spark para simular volumen en `lh_finanzas`.
Desplegado por `fabdemo deploy`. Ejecútalo con `fabdemo data notebook` o desde el portal.

In [ ]:
# Parámetros (los placeholders los rellena fabdemo al desplegar)
filas = 1_000_000
dias = 30
tabla = "facturas_masivas"
workspace_id = "{{state.lakehouse_ws_id}}"
lakehouse_id = "{{state.lakehouse_id}}"

In [ ]:
from pyspark.sql import functions as F

regiones = F.array(*[F.lit(r) for r in ["Norte", "Sur", "Este", "Oeste", "Centro"]])
estados = F.array(*[F.lit(e) for e in ["Emitida", "Cobrada", "Vencida", "Anulada"]])

df = (spark.range(filas)
      .withColumn("cliente_n", (F.rand(7) * 400 + 1).cast("int"))
      .withColumn("cliente_id", F.format_string("C%05d", F.col("cliente_n")))
      .withColumn("fecha", F.date_sub(F.current_date(), (F.rand(11) * dias).cast("int")))
      .withColumn("importe", F.round(F.exp(F.randn(3) * 0.9 + 6), 2))
      .withColumn("iva", F.round(F.col("importe") * 0.21, 2))
      .withColumn("estado", F.element_at(estados, (F.rand(5) * 4 + 1).cast("int")))
      .withColumn("region", F.element_at(regiones, (F.col("cliente_n") % 5 + 1).cast("int")))
      .withColumn("factura_id", F.concat(F.lit("M"), F.col("id").cast("string")))
      .drop("id", "cliente_n"))

ruta = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{tabla}"
df.write.format("delta").mode("overwrite").save(ruta)
print(f"{filas:,} filas escritas en {tabla}")

In [ ]:
mssparkutils.notebook.exit(f"ok:{filas}")